# TensorIR 简介

来源：[TensorIR系列【一】 背景与简介](https://zhuanlan.zhihu.com/p/433540150)

为了充分利用硬件特性，通常需要在张量计算程序(Tensor Program)中添加张量计算指令(Tensor Intrinsic)。这种带有张量计算指令的程序称为**张量化程序**(Tensorized Program)。

TensorIR 的设计初衷就是为了解决 Tensorized Program 的编译优化问题。编译优化问题分为三个方面：

1. 怎么写 Tensorized Program
2. 怎么优化 Tensorized Program
3. 怎么把同一个 Tensorized Program 部署在不同的硬件后端上

对此，TensorIR 也针对性地提供了解决方案：

- `TVMScript`：用基于 Python AST 的语法来 TensorIR 从而表达 Tensorized Program
- 交互式 Schedule：通过 Schedule Primitive 实时优化 Tensorized Program
- 独立的 Schedule Primitive：方便开发者针对不同硬件添加特殊的 Primitive

TensorIR 采取了全新的编程范式：手写算子+Schedule。这种方式结合了手写算子的表达能力，以及（Auto）Schedule 带来的便利。用户可以根据算子的计算方式，以及后端指令的需要，先写出能够正确运行的程序，再借助 Schedule 的能力去提升性能。针对一些简单算子（如gemm），Tensor Expression 已经能够很好地表达了，TensorIR 也支持了从 TE 导入的能力。

## `TVMScript`

`TVMScript` 是一种基于 Python 语法的脚本语言（类似 `torchscript`），用来表达 TVM TensorIR。这是一种 round-trippable 的语法，这意味着不但可以从 TensorIR 打印成 TVMScript，也可以随时将 TVMScript Parse 回 IR 本身。需要注意的是，TVMScript 仅仅使用了 Python 的语法和 AST，并没有通过 Python Interpreter，因此所有在 TVMScript 外定义的变量均无法在 TVMScript 内部解析。

## 针对 Tensorized Program 的交互式优化

在 Schedule 层面，TensorIR相较于现有TVM TE Schedule有了两个主要的改进点：Tensorized Program 支持和交互式 Schedule

### Imperative Scheduling（交互式Schedule）

TE Schedule是由Halide IR改进而来，其核心是基于Schedule Tree的静态声明模式，在初期通过声明一系列Primitives，并在最后生成代码的时候一起把目标IR输出。TensorIR采用的是实时动态的变换模式，每一个Primitive即时作用于IR本身，修改IR。
这类似于Tensorflow 1.x（静态）和PyTorch（动态）的区别，我们相信Imperative Scheduling会给开发者和研究者带来更好的易用性。

### 原生的Tensorized Program支持

Halide 从设计本身就是基于标量计算设计的优化工具，虽然 TVM 在其基础上增加了和 Tensorize 的支持。但其用的是匹配替换的方式做的支持，使得 Tensorize 之后的代码就无法再做任何修改，大大增加了 Tensorized Program 的优化难度。TensorIR 将张量计算作为 First-class citizen，做到了原生支持。在 Schedule 的任何阶段，都可以对任何张量计算做变换和优化。

## TensorIR 独立 Primitive

TensorIR 针对 CPU 和 GPU 提供了绝大多数常用的 Primitive。通过这些 Primitive 能够生成充分利用硬件性能的程序（目前已经实现了与 Cutlass 齐平的性能）。当然，针对其他加速器，这些 Primitive 通常来说是不够的，需要额外添加 Schedule Primitive 也是常有的情况。

TensorIR 提供了完全独立的 Primitive 实现方式，使其不依赖于任何独立数据结构，仅仅通过 IRModule 本身就能完成 Primitive 的实现（类似 Pass）。下面是典型的 Primitive 的实现方式：

1. 根据现有 IR 检查合法性
2. 基于现有 IR 生成希望得到的部分 IR
3. 使用已经封装好的 Replace 接口，实现 AST 的替换

```cpp
StmtSRef ExamplePrimitive(ScheduleState self, ...) {
  // Step 1. Check correctness
  assert CheckValidation(self, old_stmt);
  // Step 2. Create wanted Stmt
  Stmt new_stmt = Mutate(old_stmt);
  // Step 3. Replace
  self->Replace(old_stmt, new_stmt);
}
```

## 张量程序抽象：TensorIR

In [1]:
import sys
sys.path.append("../../")
import set_env

ROOT: /media/pc/data/lxw/ai/tvm-book


In [2]:
import numpy as np
import tvm
from tvm.ir.module import IRModule
from tvm.script import ir as I, tir as T

In [3]:
dtype = "float32"
a_np = np.random.rand(100, 128).astype(dtype)
b_np = np.random.rand(128, 80).astype(dtype)
c_mm_relu = np.maximum(a_np @ b_np, 0)

为了显式描述计算过程，编写如下函数：

In [4]:
def lnumpy_mm_relu(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    Y = np.empty((100, 80), dtype="float32")
    for i in range(100):
        for j in range(80):
            for k in range(128):
                if k == 0:
                    Y[i, j] = 0
                Y[i, j] = Y[i, j] + A[i, k] * B[k, j]
    for i in range(100):
        for j in range(80):
            C[i, j] = max(Y[i, j], 0)

In [5]:
c_np = np.empty((100, 80), dtype=dtype)
lnumpy_mm_relu(a_np, b_np, c_np)
np.testing.assert_allclose(c_mm_relu, c_np, rtol=1e-5)

In [10]:
@tvm.script.ir_module
class MyModule:
    @T.prim_func
    def mm_relu(A: T.Buffer((100, 128), "float32"),
                B: T.Buffer((128, 80), "float32"),
                C: T.Buffer((100, 80), "float32")):
        T.func_attr({"global_symbol": "mm_relu", "tir.noalias": True})
        Y = T.alloc_buffer((100, 80), dtype="float32")
        for i, j, k in T.grid(100, 80, 128):
            with T.block("Y"):
                vi, vj, vk = T.axis.remap("SSR", [i, j, k])
                with T.init():
                    Y[vi, vj] = T.float32(0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(100, 80):
            with T.block("C"):
                vi, vj = T.axis.remap("SS", [i, j])
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0))

In [13]:
MyModule.show()

In [14]:
sch = tvm.tir.Schedule(MyModule)

In [15]:
block_Y = sch.get_block("Y", func_name="mm_relu")
i, j, k = sch.get_loops(block_Y)

In [16]:
j0, j1 = sch.split(j, factors=[None, 4]) # 将循环 j 分成两个循环，其中内部循环的长度为 4

In [17]:
sch.mod.show()

In [18]:
sch.reorder(j0, k, j1) # 重新排序循环

In [19]:
sch.mod.show()

In [20]:
# 将块 C 移动到 Y 的内循环里
block_C = sch.get_block("C", "mm_relu")
sch.reverse_compute_at(block_C, j0)
sch.mod.show()

In [21]:
# Y 元素的初始化与归约更新分开
sch.decompose_reduction(block_Y, k)
sch.mod.show()